<a href="https://colab.research.google.com/github/FRA-0023/Causal-Networks/blob/main/notebooks/scm_matrix_form.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture # Capture and suppress the output of the cell.
!apt install python3-dev graphviz libgraphviz-dev pkg-config # Install system dependencies for graphviz, which is used for graph visualization.
!pip install numpy scipy pandas networkx pygraphviz matplotlib # Install Python packages required for numerical operations, scientific computing, data manipulation, graph creation, graph visualization, and plotting.

UsageError: unrecognized arguments: Capture and suppress the output of the cell.


In [2]:
import numpy as np # Import the NumPy library for numerical operations, aliased as 'np'.
import scipy as sp # Import the SciPy library for scientific computing, aliased as 'sp'.
import pandas as pd # Import the Pandas library for data manipulation and analysis, aliased as 'pd'.
import networkx as nx # Import the NetworkX library for creating and manipulating graphs, aliased as 'nx'.
import matplotlib.pyplot as plt # Import the Matplotlib library's pyplot module for plotting, aliased as 'plt'.

from scipy.optimize import minimize # Import the 'minimize' function from SciPy's optimization module, used for finding the minimum of a function.
from networkx.drawing.nx_agraph import graphviz_layout # Import 'graphviz_layout' for generating graph layouts using Graphviz, which provides better visual organization for complex graphs.
from typing import List, Tuple # Import 'List' and 'Tuple' from the 'typing' module for type hinting, improving code readability and maintainability.

# Structural Causal Models: Matrix Form

By Alessio Zanga, Niccolò Rocchi and Fabio Stella

## Abstract

There are various types of Structural Causal Models (SCMs) with their associated functional form:

|          Type          |                 Functional Form                   |
|------------------------|---------------------------------------------------|
| General SCM            | $$X_i := f_{X_i}(Pa(X_i), U_{X_i})$$              |
| Additive Noise Model   | $$X_i := f_{X_i}(Pa(X_i)) + U_{X_i}$$             |
| Linear Gaussian Model  | $$X_i := \sum_{X_j \in Pa(X_i)} \beta_{ij} X_j + c_i + \mathcal{N}(\mu_i, \sigma_i^2)$$                                                          |

We are going to model a **Linear Gaussian** SCM, a particular version of additive models, due to its simplicity and flexibility.

Observe that, by incorporating the intercept $c_i$ into $\mu_i$, we may assume without loss of generality that $c_i = 0$. However, in the whole notebook we'll assume **zero mean** for the noises, so that the Linear Gaussian SCM takes the form:

$$
X_i := \sum_{X_j \in Pa(X_i)} \beta_{ij} X_j + \mathcal{N}(0, \sigma_i^2)
$$

## Introduction

### Systems of Linear Equations

Recall the definition of a system of linear equations:

$\begin{cases}
  a_{11} x_1 + a_{12} x_2 + \dots + a_{1n} x_n = b_1 \\
  a_{11} x_1 + a_{12} x_2 + \dots + a_{1n} x_n = b_2 \\
  \vdots \\
  a_{mn} x_1 + a_{mn} x_2 + \dots + a_{mn} x_n = b_m \\
\end{cases}$

or more concisely as:

$x_1 \begin{bmatrix}
a_{11} \\
a_{21} \\
\vdots \\
a_{m1}
\end{bmatrix} +
x_2 \begin{bmatrix}
a_{12} \\
a_{22} \\
\vdots \\
a_{m2}
\end{bmatrix} +
\dots +
x_n \begin{bmatrix}
a_{1n} \\
a_{2n} \\
\vdots \\
a_{mn}
\end{bmatrix} =
\begin{bmatrix}
b_{1} \\
b_{2} \\
\vdots \\
b_{m}
\end{bmatrix}$

which can be expressed in **matrix form** as:

$\underset{(m \times n)}{\mathbf{A}} \underset{(n \times 1)}{\mathbf{x}} = \underset{(m \times 1)}{\mathbf{b}}$

where:

* $\mathbf{A}$ is the matrix of *coefficients*,
* $\mathbf{x}$ is a vector of *unknowns*,
* $\mathbf{b}$ is a vector of *constant terms*.

In case where there is a number of equations $m$ equals to the number of unknowns $n$, we have:

$\underset{({\color{red} n} \times n)}{\mathbf{A}} \underset{(n \times 1)}{\mathbf{x}} = \underset{({\color{red} n} \times 1)}{\mathbf{b}}$

hence, $\mathbf{A}$ is a square matrix.

### Defining an SCM with Matrix Form

We are going to employ the matrix form of a system of linear equations to model a **Linear Gaussian** SCM $\mathcal{M}$.

Recall that an SCM $\mathcal{M}$ is defined by a tuple $(\mathbf{V}, \mathbf{U}, \mathbf{F})$, where:


* $\mathbf{V}$ is the set of $n$ endogenous variables,
* $\mathbf{U}$ is the set of $m$ exogenous variables,
* $\mathbf{F}$ is the set of $n$ functions, one for each $V_i$.

If $\mathbf{F}$ contains only linear functions, we can express it as a **system of linear equations**.

Moreover, in a **Linear Gaussian** SCM, the exogenous variables are independent among each other and Gaussian distributed. Moreover, each noise $\mathcal{N}(0, \sigma^2_i)$ is a linear combination of a subset $\mathbf{U}_i \subseteq \mathbf{U}$. Remember indeed that a linear combination of Gaussian distributions is still Gaussian.

During the modelling step, a first pitfall would be to map the variables in $\mathbf{V} \cup \mathbf{U}$ directly to the unknowns $\mathbf{x}$. This will result in a $\mathbf{A}$ coefficient square matrix with shape $([n+m], [n+m])$, with $|\mathbf{V}| = n$ and $|\mathbf{U}| = m$. It would result in a system of the form:

$\underset{([n + m] \times [n + m])}{\mathbf{A}} \underset{([n + m] \times 1)}{\mathbf{x}} = \underset{([n + m] \times 1)}{\mathbf{b}}$

However, we cannot observe the noises, that is the values they assume.

Therefore, we map:

* the set of endogenous variables $\mathbf{V}$ to endogneous unknowns $\mathbf{x}$ and,
* the set of exogenous variables $\mathbf{U}$ to exogenous unknowns $\mathbf{y}$.

This means to exploit the endogeneous unknowns $\mathbf{x}$, the exogenous unknowns $\mathbf{y}$ and the inherited *additive noise model* as:

$\underset{(n \times n)}{\vphantom{y}\mathbf{A}}
 \underset{(n \times 1)}{\vphantom{y}\mathbf{x}}
 {\color{red}{- \underset{(n \times m)}{\vphantom{y}\pmb{\Gamma}}
 \underset{(m \times 1)}{\mathbf{y}}}} =
 \underset{({\color{red} n} \times 1)}{\vphantom{y}\mathbf{b}}$

The meaning of the minus ($-$) sign will be clear in a moment.

Please, notice how the shape of $\mathbf{b}$ is only $(n \times 1)$ and not $([n + m] \times 1)$, since we cannot observe directly the actual value of the noise terms $\mathbf{y}$, but only their effect on $\mathbf{x}$, which is hidden behind the sum operation.

For example, consider the following SCM $\mathcal{M}$:

$\begin{cases}
  x_1 := 2x_2 + y_1 \\
  x_2 := 3x_3 + y_2 \\
  x_3 := 2y_2 + y_3
\end{cases}$

Then we can express the SCM in the language of a system of linear equations:

$\begin{cases}
  x_1 - 2x_2 - y_1 = 0 \\
  x_2 - 3x_3 - y_2 = 0 \\
  x_3 - 2y_2 \, - y_3 = 0
\end{cases}$

which is explicitly:

$\begin{cases}
  {\color{red} 1}x_1 - 2x_2 \,{\color{red} {+ \; 0x_3 - 1}}y_1 \, {\color{red} {+ \; 0y_2 + 0y_3}} = 0 \\
  {\color{red} {0x_1 + 1}}x_2 - 3x_3 \, {\color{red} {+ \; 0y_1 - 1}}y_2 \;{\color{red} {+ \, 0y_3}} = 0 \\
  {\color{red} {0x_1 + 0x_2 + 1}}x_3 \, {\color{red} {+ \; 0y_1}} - 2y_2 \; {\color{red} {- \, 1}}y_3 = 0
\end{cases}$

It has the following matrix form:

In [3]:
# Coefficient matrix for the endogenous variables.
A = np.array([ # Define a NumPy array named 'A' to represent the coefficient matrix for endogenous variables in a system of linear equations.
    [1., 2., 0.], # First row of the matrix, representing the coefficients for the first equation.
    [0., 1., 3.], # Second row of the matrix, representing the coefficients for the second equation.
    [0., 0., 1.], # Third row of the matrix, representing the coefficients for the third equation.
])
# Endogenous variables.
x = ["$x_1$", "$x_2$", "$x_3$"] # Define a list 'x' containing string representations of the endogenous variables in the SCM.
# Coefficient matrix for the exogenous variables.
C = np.array([ # Define a NumPy array named 'C' to represent the coefficient matrix for exogenous variables (noise terms).
    [1., 0., 0.], # First row of the matrix, showing how exogenous variables influence the first endogenous variable.
    [0., 1., 0.], # Second row of the matrix, showing how exogenous variables influence the second endogenous variable.
    [0., 2., 1.], # Third row of the matrix, showing how exogenous variables influence the third endogenous variable.
])
# Exogenous variables.
y = ["$y_1$", "$y_2$", "$y_3$"] # Define a list 'y' containing string representations of the exogenous variables (noise terms) in the SCM.
# Constant terms (zero because of zero mean of noises)
b = [0, 0, 0] # Define a list 'b' representing constant terms in the system of equations, set to zeros assuming zero-mean noises.

As you might have noticed, the diagonal of $\mathbf{A}$ is always $\mathbf{1}$, therefore, to make it explicit and to differentiate it from the standard notation used in graphs, we rewrite both $\mathbf{A}$ as:

${\color{red}{
    \underset{(n \times n)}{\vphantom{y}(\mathbf{I} - \mathbf{B})}
 }}
 \underset{(n \times 1)}{\vphantom{y}\mathbf{x}} -
 \underset{(n \times m)}{\vphantom{y}\pmb{\Gamma}}
 \underset{(m \times 1)}{\mathbf{y}} =
\underset{(n \times 1)}{\vphantom{y}\mathbf{b}}$

where $\mathbf{I}$ is the identity matrix.

Rearranging the terms, the SCM form becomes more visible:

${\color{red}{
    \underset{(n \times n)}{\vphantom{y}\mathbf{I}}
 }}
 \underset{(n \times 1)}{\vphantom{y}\mathbf{x}} =
 {\color{red}{
    \underset{(n \times n)}{\vphantom{y}\mathbf{B}}
 }}
 \underset{(n \times 1)}{\vphantom{y}\mathbf{x}} +
 \underset{(n \times m)}{\vphantom{y}\pmb{\Gamma}}
 \underset{(m \times 1)}{\mathbf{y}} +
\underset{(n \times 1)}{\vphantom{y}\mathbf{b}}$

In [4]:
# Coefficient matrix for endogenous variables.
B = np.array([ # Define a NumPy array named 'B' to represent the direct causal effects between endogenous variables.
    [0., 2., 0.], # First row of the matrix, indicating effects on the first endogenous variable.
    [0., 0., 3.], # Second row of the matrix, indicating effects on the second endogenous variable.
    [0., 0., 0.], # Third row of the matrix, indicating effects on the third endogenous variable.
])
# Identity matrix with same shape as endogenous coefficient matrix.
I = np.eye(*B.shape) # Create an identity matrix 'I' with dimensions matching the shape of matrix 'B', used for expressing (I - B) form.
# Coefficient matrix for exogenous variables.
C = np.array([ # Define a NumPy array named 'C' to represent the direct causal effects of exogenous variables (noise terms) on endogenous variables.
    [1., 0., 0.], # First row of the matrix, showing influence on the first endogenous variable.
    [0., 1., 0.], # Second row of the matrix, showing influence on the second endogenous variable.
    [0., 2., 1.], # Third row of the matrix, showing influence on the third endogenous variable.
])

### Expliciting the underlying Causal Graph

Following the causal edge assumption, $\mathbf{V}$ and $\mathbf{U}$ are vertices of a causal graph $\mathcal{G}$ induced by $\mathbf{F}$ :

$$
X_i := \sum_{X_j \in Pa(\mathcal{G}, X_i)} \beta_{ij} X_j + \mathcal{N}(0, \sigma_i^2), \quad \forall X_i \in \mathbf{V}
$$

A possible representation of directed graph $\mathcal{G}$ is by means of a squared *binary* adjacency matrix $\mathbf{A}$ :

$\mathbf{A}[i, j] =
a_{ij} =
\begin{cases}
  1 & \text{if } (X_i \rightarrow X_j) \in \mathbf{E}, \\
  0 & \text{otherwise.}
\end{cases}$

We can laverage the assumptions on the exogenous variables to build the adjacency matrix $\mathbf{A}$, specifically:

1. An endogenous variable cannot be parent of an exogenous variable, i.e. $(V \rightarrow U) \notin \mathbf{E}, \, \forall \, (V, U) \in \mathbf{V} \times \mathbf{U}$,
2. The exogenous variables are independent of each other, i.e. $(U_i \rightarrow U_j) \notin \mathbf{E}, \, \forall \, (U_i, U_j) \in \mathbf{U} \times \mathbf{U}$.

Hence, the real-valued adjacency matrix $\mathbf{W}$ is a block matrix of the form:

$\mathbf{W} = %
\left[
\begin{array}{c|c}
  \mathbf{B} & \pmb{\Gamma} \\ \hline
  \mathbf{0} & \mathbf{0} \\
\end{array}
\right]^T
= %
\left[
\begin{array}{c|c}
  \mathbf{B}^T & \mathbf{0} \\ \hline
  \pmb{\Gamma}^T & \mathbf{0} \\
\end{array}
\right]$

To obtain the binary adjacency matrix $\mathbf{A}$ we just map the previous real-valued matrix to a boolean matrix, hence by imposing $\mathbf{W} \neq 0$, and converting boolean values to $\{0, 1\}$.

Notice also we have already dismissed the notation $\mathbf{A}$ to indicate $\mathbf{I} - \mathbf{B}$. Here, $\mathbf{A}$ indicates the graph adjacency matrix, included the noise terms.

For example:

In [5]:
# Get the variables sizes.
n = len(x) # Get the number of endogenous variables by counting elements in list 'x'.
m = len(y) # Get the number of exogenous variables by counting elements in list 'y'.
# Compute the adjacency matrix.
W = np.vstack([B.T, C.T]) # Stack the transpose of matrix B (endogenous effects) and C (exogenous effects) vertically to form an intermediate matrix W.
W = np.hstack([W, np.zeros((n + m, m))]) # Stack W horizontally with a zero matrix of appropriate size to complete the full weighted adjacency matrix structure, accommodating both endogenous and exogenous variables.
A = (W != 0) # Create a boolean adjacency matrix 'A' by checking which elements of 'W' are non-zero, representing the presence of an edge.

print(A) # Print the resulting binary adjacency matrix 'A'.

[[False False False False False False]
 [ True False False False False False]
 [False  True False False False False]
 [ True False False False False False]
 [False  True  True False False False]
 [False False  True False False False]]


We can now build the causal graph $\mathcal{G}$ given the adjacency matrix $\mathbf{A}$ :

In [6]:
def graph_from_matrix(x: List[str], y: List[str], A: np.array) -> nx.DiGraph:
    # Compute the directed graph given the adjacency matrix.
    G = nx.from_numpy_array(A, create_using=nx.DiGraph) # Create a directed graph 'G' from the NumPy adjacency array 'A', where 'A' defines the connections.
    # Set nodes labels.
    G = nx.relabel_nodes(G, dict(zip(G.nodes, x + y))) # Relabel graph nodes. The original integer node labels are replaced with meaningful string labels from the combined list of endogenous (x) and exogenous (y) variables.
    # Set exogenous and endogenous variables as attributes.
    G.graph["V"] = set(x) # Store the set of endogenous variables 'x' as a graph attribute named "V" for easy access.
    G.graph["U"] = set(y) # Store the set of exogenous variables 'y' as a graph attribute named "U" for easy access.

    return G # Return the constructed directed graph 'G' with labeled nodes and variable sets as attributes.

Let's define a plot style:

In [7]:
def draw(G): # Define a function 'draw' that takes a NetworkX graph 'G' as input for visualization.
    plt.figure() # Create a new figure to draw the graph on.
    pos = graphviz_layout(G, prog = "dot") # Compute node positions using the Graphviz 'dot' layout algorithm, which is suitable for directed graphs, to arrange nodes aesthetically.
    for v in G.nodes(): # Iterate through each node 'v' in the graph 'G'.
        nx.draw_networkx_nodes( # Draw the nodes of the graph.
            G, # The graph object.
            pos = pos, # The pre-computed positions for all nodes.
            nodelist = [v], # A list containing only the current node 'v' to draw it individually.
            node_color = "white", # Set the background color of the node to white.
            edgecolors = "black" if v in G.graph["V"] else "gray", # Set the border color of the node to black if it's an endogenous variable, otherwise gray for exogenous variables.
            node_size = 500, # Set the size of the nodes for better visibility.
            node_shape = "o", # Set the shape of the nodes to a circle.
        )
        nx.draw_networkx_labels( # Draw the labels for the nodes.
            G, # The graph object.
            pos = pos, # The same pre-computed positions for node labels.
            font_size = 15, # Set the font size for node labels.
        )
    for (u, v) in G.edges(): # Iterate through each edge (u, v) in the graph 'G'.
        nx.draw_networkx_edges( # Draw the edges of the graph.
            G, # The graph object.
            pos = pos, # The pre-computed positions for edges.
            edgelist = [(u, v)], # A list containing only the current edge (u, v) to draw it individually.
            edge_color = "black", # Set the color of the edge to black.
            style = "solid" if u not in G.graph["U"] else "dashed", # Set the style of the edge: solid if the source node 'u' is endogenous, dashed if 'u' is exogenous (representing noise/error).
            width = 1.0, # Set the width of the edges.
        )
    weights = nx.get_edge_attributes(G, "weight") # Retrieve the 'weight' attribute for all edges in the graph, if they exist.
    if type(list(weights.values())[0]) != bool: # Check if the edge weights are numerical (not boolean, which would imply unweighted edges).
        nx.draw_networkx_edge_labels( # Draw labels for the edges, typically showing their weights.
            G, # The graph object.
            pos = pos, # The pre-computed positions for edge labels.
            edge_labels = weights, # A dictionary mapping edges to their weights, used as labels.
            font_color = "red", # Set the font color for edge labels to red.
            font_size = 13, # Set the font size for edge labels.
        )

In [8]:
# Build the causal graph from adjacency matrix.
G = graph_from_matrix(x, y, A) # Call the 'graph_from_matrix' function to construct the causal graph 'G' using the endogenous variables 'x', exogenous variables 'y', and the binary adjacency matrix 'A'.
# Plot the causal graph.
draw(G) # Call the 'draw' function to visualize the newly created causal graph 'G'.

ImportError: requires pygraphviz http://pygraphviz.github.io/

<Figure size 640x480 with 0 Axes>

If we want to make the coefficient explicit, we can build a **weighted** directed graph:

In [9]:
# Build the causal graph from adjacency matrix.
G = graph_from_matrix(x, y, W) # Call the 'graph_from_matrix' function to construct the causal graph 'G' using the endogenous variables 'x', exogenous variables 'y', and the weighted adjacency matrix 'W'. This will include edge weights.
# Plot the causal graph.
draw(G) # Call the 'draw' function to visualize this weighted causal graph 'G'.

ImportError: requires pygraphviz http://pygraphviz.github.io/

<Figure size 640x480 with 0 Axes>

## Implementation

### Sampling from an SCM

Suppose we are given a SCM $\mathcal{M}$. How do we sample a data set $\mathbf{D}$ from $\mathcal{M}$ ?

Keep in mind we are assuming each noise to have a $0$ mean. Since we are considering a linear Gaussian SCM $\mathcal{M}$, sampling the data set $\mathbf{B}$ from $\mathcal{M}$ is equivalent to sampling from a **multivariate normal distribution** $\mathcal{N}(0, \pmb{\Sigma})$, where the variables in $\mathcal{N}$ are $\mathbf{V} \cup \mathbf{U}$, and $\pmb{\Sigma}$ is their covariance matrix.

With these assumptions, the covariance matrix over $\mathbf{V} \cup \mathbf{U}$ is defined as:

$\pmb{\Sigma} = (\mathbf{I} - \mathbf{B})^{-1}
\pmb{\Psi}
\left((\mathbf{I} - \mathbf{B})^{-1}\right)^{T}$

where $\pmb{\Psi} = \pmb{\Gamma} + \pmb{\Gamma}^T - \mathbf{I}\cdot\pmb{\Gamma}$, i.e. the symmetric of $\pmb{\Gamma}$.

Hence, we can define a sampling procedure by leveraging $\mathbf{B}$ and $\pmb{\Gamma}$:

In [10]:
def sample(M: Tuple, size: int = int(1e6), seed: int = 31) -> pd.DataFrame:
    # Explicit matrix form.
    (x, y, B, C) = M # Unpack the tuple 'M' (representing the SCM) into its constituent parts: endogenous variable names (x), exogenous variable names (y), the B matrix (endogenous effects), and the C matrix (exogenous effects).
    # Allocate the identity matrix.
    I = np.eye(*B.shape) # Create an identity matrix 'I' with the same dimensions as matrix 'B'. This is used in the formula (I - B).
    # Compute the scale.
    IB = np.linalg.inv(I - B) # Compute the inverse of the matrix (I - B). This term scales the effects of exogenous variables to produce endogenous variable distributions.
    # Compute the actual noise distribution.
    Psi = C + C.T - np.eye(*C.shape) * C # Compute the Psi matrix, which represents the variance-covariance matrix of the exogenous (noise) variables. The formula ensures it's symmetric.
    # Compute the actual data distribution.
    Sigma = IB @ Psi @ IB.T # Compute the model-implied covariance matrix 'Sigma' for the endogenous variables. This formula is derived from the linear Gaussian SCM equations.
    # Set the seed for reproducibility.
    np.random.seed(seed) # Set the seed for NumPy's random number generator to ensure that the sampling process is reproducible.
    # Sample from a multivariate normal distribution.
    D = np.random.multivariate_normal([0] * B.shape[0], Sigma, size) # Sample 'size' data points from a multivariate normal distribution. The mean vector is zero (since noises have zero mean), and the covariance matrix is 'Sigma'.
    # Compute the data frame.
    D = pd.DataFrame(data = D, columns = x) # Create a Pandas DataFrame 'D' from the sampled data. The columns are named according to the endogenous variables 'x'.

    return D # Return the sampled DataFrame 'D'.

We can now sample from a model:

In [11]:
# Specify the model.
M = (x, y, B, C) # Create a tuple 'M' that encapsulates the structural causal model's components: endogenous variable names (x), exogenous variable names (y), the B matrix, and the C matrix.
# Sample the data.
D = sample(M) # Call the 'sample' function, passing the model 'M', to generate a DataFrame 'D' containing simulated data from the SCM.
# Compute the summary statistics.
D.describe() # Display descriptive (summary) statistics for each column in the sampled DataFrame 'D', such as mean, standard deviation, min, max, and quartiles.

/tmp/ipykernel_385/2808761900.py:15: RuntimeWarning: covariance is not symmetric positive-semidefinite.
  D = np.random.multivariate_normal([0] * B.shape[0], Sigma, size) # Sample 'size' data points from a multivariate normal distribution. The mean vector is zero (since noises have zero mean), and the covariance matrix is 'Sigma'.


,$x_1$,$x_2$,$x_3$
count,1000000.000000,1000000.000000,1000000.000000
mean,0.009500,0.004184,0.001467
std,9.430900,4.689909,1.120898
min,-43.815576,-21.966539,-5.664769
25%,-6.355846,-3.160213,-0.754233
50%,0.003506,0.002797,0.000623
75%,6.379702,3.170067,0.758427
max,46.216736,22.477628,5.369341


### Fitting an SCM

Suppose we are given a data set $\mathbf{D}$ and a causal graph $\mathcal{G}$. How do we recover the associated SCM $\mathcal{M}$ given $\mathbf{D}$ and $\mathcal{G}$ ?

Recall that the underlying system of SCM $\mathcal{M}$ was:

$\begin{cases}
  {\color{red} 1}x_1 - 2x_2 \,{\color{red} {+ \; 0x_3 - 1}}y_1 \, {\color{red} {+ \; 0y_2 + 0y_3}} = b_1 \\
  {\color{red} {0x_1 + 1}}x_2 - 3x_3 \, {\color{red} {+ \; 0y_1 - 1}}y_2 \;{\color{red} {+ \, 0y_3}} = b_2 \\
  {\color{red} {0x_1 + 0x_2 + 1}}x_3 \, {\color{red} {+ \; 0y_1}} - 2y_2 \; {\color{red} {- \, 1}}y_3 = b_3
\end{cases}$

Recovering the SCM $\mathcal{M}$ would require to estimate the unknown parameters ${\color{blue} {\beta_{ij}}}$ and ${\color{blue} {\gamma_{ij}}}$ :

$\begin{cases}
  {\color{blue} {\beta_{11}}}x_1 + {\color{blue} {\beta_{12}}}x_2 + {\color{blue} {\beta_{13}}}x_3 + {\color{blue} {\gamma_{11}}}y_1 + {\color{blue} {\gamma_{12}}}y_2 + {\color{blue} {\gamma_{13}}}y_3 = b_1 \\
  {\color{blue} {\beta_{21}}}x_1 + {\color{blue} {\beta_{22}}}x_2 + {\color{blue} {\beta_{23}}}x_3 + {\color{blue} {\gamma_{21}}}y_1 + {\color{blue} {\gamma_{22}}}y_2 + {\color{blue} {\gamma_{23}}}y_3 = b_2 \\
  {\color{blue} {\beta_{31}}}x_1 + {\color{blue} {\beta_{32}}}x_2 + {\color{blue} {\beta_{33}}}x_3 + {\color{blue} {\gamma_{31}}}y_1 + {\color{blue} {\gamma_{32}}}y_2 + {\color{blue} {\gamma_{33}}}y_3 = b_3 \\
\end{cases}$

more concisely:

$(\mathbf{I} - \mathbf{B}) =
\begin{bmatrix}
  {\color{blue} {\beta_{11}}} & {\color{blue} {\beta_{12}}} & {\color{blue} {\beta_{13}}} \\
  {\color{blue} {\beta_{21}}} & {\color{blue} {\beta_{22}}} & {\color{blue} {\beta_{23}}} \\
  {\color{blue} {\beta_{31}}} & {\color{blue} {\beta_{32}}} & {\color{blue} {\beta_{33}}}
\end{bmatrix},
~ - \pmb{\Gamma} =
\begin{bmatrix}
  {\color{blue} {\gamma_{11}}} & {\color{blue} {\gamma_{12}}} & {\color{blue} {\gamma_{13}}} \\
  {\color{blue} {\gamma_{21}}} & {\color{blue} {\gamma_{22}}} & {\color{blue} {\gamma_{23}}} \\
  {\color{blue} {\gamma_{31}}} & {\color{blue} {\gamma_{32}}} & {\color{blue} {\gamma_{33}}}
\end{bmatrix}$

If we constrain $\mathbf{B}$ and $\pmb{\Gamma}$ to follow the structure implied by the graph $\mathcal{G}$, then we have:

In [12]:
# Compute the adjacency matrix.
A = nx.adjacency_matrix(G).todense().astype(bool).astype(float) # Obtain the adjacency matrix from the NetworkX graph 'G', convert it to a dense NumPy array, then to boolean (where 0 means no edge and non-zero means an edge), and finally to float.
# Get the number of endogenous and exogenous variables.
n = len(G.graph["V"]) # Get the number of endogenous variables from the "V" attribute stored in the graph 'G'.
m = len(G.graph["U"]) # Get the number of exogenous variables from the "U" attribute stored in the graph 'G'.
# Extract the direct effects of endogenous variables, i.e. (Beta != 0).
B_0 = A[0:n, 0:n].T.copy() # Extract the submatrix from 'A' that corresponds to connections between endogenous variables (V to V). This submatrix represents the structural zeros for the B matrix. It's transposed because NetworkX adjacency matrix has (target, source) while B is (source, target).
# Extract the variances-covariances of exogenous variables, i.e. (Gamma != 0).
C_0 = A[n:, 0:n].T.copy() # Extract the submatrix from 'A' that corresponds to connections from exogenous variables (U) to endogenous variables (V). This submatrix represents the structural zeros for the Gamma (C) matrix, also transposed.

# Compute non-zero entries of Beta_0.
k = int(np.sum(B_0)) # Calculate 'k', the total number of non-zero entries in B_0. This 'k' indicates how many parameters in the B matrix need to be estimated.

print(B_0) # Print the B_0 matrix, showing the structural pattern (where parameters exist) for endogenous variable effects.
print(C_0) # Print the C_0 matrix, showing the structural pattern (where parameters exist) for exogenous variable effects.

[[0. 1. 0.]
 [0. 0. 1.]
 [0. 0. 0.]]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 1. 1.]]


Hence, the only unknown parameters are:

$(\mathbf{I} - \mathbf{B}) =
\begin{bmatrix}
  1 & {\color{blue} {\beta_{12}}} & 0 \\
  0 & 1 & {\color{blue} {\beta_{23}}} \\
  0 & 0 & 1
\end{bmatrix},
~ - \pmb{\Gamma} =
\begin{bmatrix}
  {\color{blue} {\gamma_{11}}} & 0 & 0 \\
  0 & {\color{blue} {\gamma_{22}}} & 0 \\
  0 & {\color{blue} {\gamma_{32}}} & {\color{blue} {\gamma_{33}}}
\end{bmatrix}$

which can be represented as a **row vector** of unknown parameters:

$\theta =
\left[
\begin{array}{cc|cccc}
  {\color{blue} {\beta_{12}}} &
  {\color{blue} {\beta_{23}}} &
  {\color{blue} {\gamma_{11}}} &
  {\color{blue} {\gamma_{22}}} &
  {\color{blue} {\gamma_{32}}} &
  {\color{blue} {\gamma_{33}}}
\end{array}
\right]$

Define the **model implied** covariance matrix $\pmb{\Sigma}(\hat{\theta})$ given the estimated parameters $\hat{\theta}$:

$\pmb{\Sigma}(\hat{\theta}) = (\mathbf{I} - \mathbf{B}(-\hat{\theta}))^{-1}
\pmb{\Psi}(-\hat{\theta})
\left((\mathbf{I} - \mathbf{B}(-\hat{\theta}))^{-1}\right)^{T}$

Computing $\mathbf{B}(-\hat{\theta})$ and $\pmb{\Gamma}(-\hat{\theta})$ only requires to **assign** $-\hat{\theta}$ to non-zero entries of such matrices.

Let's define an assignment function:

In [13]:
def assign(X: np.ndarray, x: np.ndarray) -> np.ndarray:
    # Make a copy of X.
    X = X.copy() # Create a copy of the input NumPy array 'X' to ensure that the original array is not modified by this function.
    # Get non-zero indices.
    i = np.ravel(X).nonzero()[0] # Flatten the array 'X' using `np.ravel()`, then find the indices of all non-zero elements using `nonzero()`. The `[0]` extracts the array of indices.
    # Assign non-zero values.
    X.flat[i] = x # Assign the values from the input array 'x' to the positions in 'X' that were previously identified as non-zero. `X.flat` provides a flat iterator over the array.

    return X # Return the modified array 'X' with the new values assigned to its non-zero positions.

For example:

In [14]:
# Solution for the above system.
theta = np.array([-2., -3., -1., -1., -2., -1.]) # Define a NumPy array 'theta' containing a set of example parameter values. These represent the true (or assumed true) coefficients for the SCM.

print(assign(B_0, np.negative(theta[:k]))) # Call the 'assign' function: it takes B_0 (structural zeros for endogenous effects) and assigns the negative of the first 'k' parameters from 'theta' to its non-zero positions, then prints the resulting matrix. This simulates filling the B matrix.
print(assign(C_0, np.negative(theta[k:]))) # Call the 'assign' function: it takes C_0 (structural zeros for exogenous effects) and assigns the negative of the remaining parameters from 'theta' to its non-zero positions, then prints the resulting matrix. This simulates filling the C matrix.

[[0. 2. 0.]
 [0. 0. 3.]
 [0. 0. 0.]]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 2. 1.]]


Compute the covariance matrix $\pmb{\Sigma}(\hat{\theta})$ given the estimated parameters $\hat{\theta}$:

In [15]:
def Sigma(theta: np.ndarray) -> np.ndarray:
    # Assign the negation of estimated parameters theta to B.
    B = assign(B_0, np.negative(theta[:k])) # Create the 'B' matrix by assigning the negative of the first 'k' parameters from 'theta' to the non-zero positions defined by 'B_0'. This fills in the endogenous causal coefficients.
    # Allocate the identity matrix.
    I = np.eye(*B.shape) # Create an identity matrix 'I' with the same dimensions as the 'B' matrix.
    # Compute the scale.
    IB = np.linalg.inv(I - B) # Compute the inverse of the matrix (I - B). This term is crucial for propagating causal effects from noises to observed variables.

    # Assign the negation of estimated parameters theta to Gamma.
    C = assign(C_0, np.negative(theta[k:])) # Create the 'C' matrix (representing Gamma) by assigning the negative of the remaining parameters from 'theta' to the non-zero positions defined by 'C_0'. This fills in the exogenous causal coefficients.
    # Compute the variances and covariances of the exogenous variables.
    Psi = C + C.T - np.eye(*C.shape) * C # Compute the Psi matrix, which represents the variance-covariance matrix of the exogenous (noise) variables. The formula ensures it is symmetric.

    # Enforce positivity.
    np.fill_diagonal(Psi, np.exp(np.diagonal(Psi))) # To ensure that variances (diagonal elements of Psi) are strictly positive, we apply an exponential transformation to them. This is often done during optimization to constrain parameters to positive values.

    # Compute the *model implied* covariance matrix.
    return IB @ Psi @ IB.T # Calculate and return the model-implied covariance matrix 'Sigma'. This is a core step in structural equation modeling, relating the causal parameters to the observed data's covariance structure.

In [16]:
# Print the model-implied true covariance matrix
Sigma(theta) # Call the 'Sigma' function with the predefined 'theta' (representing the true parameters) to compute and print the model-implied covariance matrix based on these parameters.

array([[159.44955497,  78.36563657,  20.30969097],
       [ 78.36563657,  39.18281828,  10.15484549],
       [ 20.30969097,  10.15484549,   2.71828183]])

Compute the observed covariance matrix $\mathbf{S}$ given the data set $\mathbf{D}$:

In [17]:
# Compute the sample covariance matrix.
S = np.cov(D.to_numpy().T) # Compute the sample covariance matrix 'S' from the sampled data 'D'. The `.to_numpy().T` converts the Pandas DataFrame to a NumPy array and then transposes it, as `np.cov` expects variables as rows.

print(S) # Print the calculated sample covariance matrix 'S'.

[[88.94187022 43.97595814  9.98538505]
 [43.97595814 21.99524571  4.95808053]
 [ 9.98538505  4.95808053  1.25641161]]


Define a *loss function* between the observed covariance matrix $\mathbf{S}$ and the model implied covariance matrix $\pmb{\Sigma}(\hat{\theta})$:

$\mathcal{L}(\mathbf{S}, \pmb{\Sigma}(\hat{\theta})) = tr(\mathbf{S}\pmb{\Sigma}(\hat{\theta})^{-1}) + \ln|\pmb{\Sigma}(\hat{\theta})| - \ln|\mathbf{S}| - n$

In [18]:
# Define loss function.
def loss(theta: np.ndarray) -> float: # Define a function 'loss' that takes a NumPy array 'theta' (the parameters to be optimized) and returns a single float value representing the discrepancy between observed and model-implied covariances.
    # Compute Sigma given theta parameters.
    Sigma_t = Sigma(theta) # Compute the model-implied covariance matrix 'Sigma_t' using the current set of 'theta' parameters.
    # Evaluate loss function.
    return np.trace(S @ np.linalg.inv(Sigma_t)) + np.log(np.linalg.det(Sigma_t)) - np.log(np.linalg.det(S)) - n # Calculate the fit function (often called F_ML or maximum likelihood loss) which quantifies the difference between the sample covariance matrix 'S' and the model-implied covariance matrix 'Sigma_t'. 'n' is the number of variables.

Minimize the loss function $\mathcal{L}$ w.r.t. $\mathbf{S}$ and $\pmb{\Sigma}(\hat{\theta})$:

In [19]:
# Initialize first solution.
np.random.seed(31) # Set the random seed for NumPy to ensure reproducibility of the initial parameter guess.
theta_0 = np.random.rand(int(np.sum(A))) # Initialize 'theta_0' as an array of random numbers between 0 and 1. The size of this array is determined by the total number of non-zero parameters in the adjacency matrix 'A' that need to be estimated.
# Minimize loss function.
optimal = minimize(loss, theta_0, method = "CG", tol = 1e-6) # Perform optimization to find the parameters that minimize the 'loss' function. It starts with 'theta_0', uses the Conjugate Gradient ('CG') method, and sets a tolerance for convergence.
# Get the solution *x*
theta_hat = optimal.x # Extract the optimized parameter values from the 'optimal' result object, which are the estimated coefficients 'theta_hat'.
# Assign estimated coefficients theta_hat to Beta and Gamma.
B = assign(B_0, np.negative(theta_hat[:k])) # Assign the negative of the first 'k' estimated parameters from 'theta_hat' to the structural positions of 'B_0' to form the optimized B matrix.
C = assign(C_0, np.negative(theta_hat[k:])) # Assign the negative of the remaining estimated parameters from 'theta_hat' to the structural positions of 'C_0' to form the optimized C matrix.

print(B) # Print the estimated B matrix, showing the optimized causal effects between endogenous variables.
print(C) # Print the estimated C matrix, showing the optimized causal effects from exogenous (noise) variables.

[[0.         1.99933925 0.        ]
 [0.         0.         2.35389296]
 [0.         0.         0.        ]]
[[0.0188324  0.         0.        ]
 [0.         1.72547889 0.        ]
 [0.         2.00062492 0.22826104]]


### Estimating with an SCM

Suppose we are given a SCM $\mathcal{M}$ and a causal estimand $\tau$. How do we estimate $\tau$ from $\mathcal{M}$ ?

In a linear SCM, the total effect $\tau = \mathbb{E}\left[ Y(1) - Y(0) \right]$ is computed as the sum of the effect of each directed path $\pi$ from $X$ to $Y$, where the effect of each directed path $\pi$ is obtained by multiplying each direct effect along such path:

$$
\tau = \mathbb{E}\left[ Y(1) - Y(0) \right] = \sum_{\pi \in \Pi(X, Y)}\prod_{(Z_i \rightarrow Z_j) \in \pi} \mathbf{B}[Z_i, Z_j]
$$

hence, by exploiting the weighted graph representation:

In [20]:
def total_effect(G: nx.DiGraph, X: str, Y: str) -> float: # Define a function 'total_effect' that calculates the total causal effect from a source node 'X' to a target node 'Y' in a given directed graph 'G'.
    # Initialize the total effect.
    tau = 0 # Initialize 'tau', the variable that will accumulate the total effect, to zero.
    # Iterate over the paths.
    for path in sorted(nx.all_simple_edge_paths(G, X, Y)): # Iterate through all simple directed paths (paths without repeated nodes) from node 'X' to node 'Y' in graph 'G'. Paths are sorted for consistent output.
        # Initialize the path effect.
        tau_pi = 1 # Initialize 'tau_pi', the effect along the current path, to 1. This will be multiplied by edge weights.
        # Iterate over the edges.
        for (i, j) in path: # Iterate through each edge (from node 'i' to node 'j') in the current path.
            # Reduce the path effect.
            tau_pi = tau_pi * G[i][j]["weight"] # Multiply 'tau_pi' by the 'weight' attribute of the current edge. The effect of a path is the product of its edge weights.
        # Accumulate the effect.
        tau = tau + tau_pi # Add the calculated effect of the current path ('tau_pi') to the total effect 'tau'.
    # Return the total effect.
    return tau # Return the final calculated total causal effect from 'X' to 'Y'.

For example, compute the effect from $X_1$ to $X_3$:

In [21]:
total_effect(G, "$x_3$", "$x_1$") # Call the 'total_effect' function to compute the total causal effect from the variable "$x_3$" to the variable "$x_1$" in the graph 'G', and display the result.

6.0